# 02 — Huấn luyện Mô hình Học máy — TechJobAI

**Notebook 2/3.** Huấn luyện 3 mô hình từ `data/it_jobs_processed.csv` (output của Notebook 1)
theo **quy trình đánh giá chuẩn**:

```
                    ┌──────────────────────────────────────────┐
 30K dòng có salary │  80% TRAIN                               │  20% TEST (CÔ LẬP)
 ───────────────────┤  • So sánh thuật toán bằng k-fold CV     ├──────────────────────
                    │  • Tinh chỉnh siêu tham số (GridSearchCV)│  • KHÔNG dùng để train/tune/chọn model
                    │  • Chọn model thắng theo điểm CV         │  • Chỉ đánh giá 1 LẦN DUY NHẤT
                    └──────────────────────────────────────────┘         model thắng cuộc
```

**Nguyên tắc then chốt:** tập test 20% bị *cô lập hoàn toàn* — mọi quyết định
(chọn thuật toán, chọn siêu tham số) đều dựa trên Cross-Validation **bên trong tập train**.
Test set chỉ được chạm vào đúng một lần để báo cáo độ chính xác cuối cùng, nên con số
R²/MAE cuối là ước lượng **không thiên vị** cho dữ liệu chưa từng thấy.

| Model | Thuật toán | Mục đích | Đánh giá |
|-------|-----------|----------|----------|
| Salary | XGBoost vs RandomForest (chọn bằng CV) | Dự đoán lương năm (USD) | CV trên train → test cô lập |
| Demand | RandomForestRegressor | Điểm nhu cầu tuyển dụng 0–100 | 5-fold CV trên train → test cô lập |
| Cluster | KMeans + PCA (K=5) | Phân cụm thị trường việc làm | Silhouette score (unsupervised) |

> Bản script tương đương: `python retrain_all.py`


In [ ]:
import os, joblib, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score, mean_absolute_error, silhouette_score

warnings.filterwarnings('ignore')
plt.style.use('dark_background')

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_FILE = os.path.join(BASE_DIR, 'data', 'it_jobs_processed.csv')
MODELS_DIR = os.path.join(BASE_DIR, 'models')
FIGURES_DIR = os.path.join(BASE_DIR, 'reports', 'figures')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

df = pd.read_csv(DATA_FILE)
print(f'Loaded {len(df):,} rows, {df.shape[1]} cols')

# Interaction features — bắt tương tác domain×seniority và state×seniority
df['domain_seniority'] = df['it_domain'].astype(str) + '_' + df['seniority_level'].astype(str)
df['state_seniority'] = df['state'].astype(str) + '_' + df['seniority_level'].astype(str)

features = ['num_skills', 'skill_diversity', 'skill_programming', 'skill_cloud', 'skill_ai_ml',
            'skill_database', 'skill_devops', 'skill_framework', 'skill_data_engineering',
            'skill_security', 'skill_soft_skills', 'years_experience',
            'seniority_level', 'job_type', 'state', 'it_domain',
            'domain_seniority', 'state_seniority']
numeric_features = [f for f in features if f.startswith('skill_') or f in ('num_skills', 'years_experience')]
categorical_features = ['seniority_level', 'job_type', 'state', 'it_domain', 'domain_seniority', 'state_seniority']

# years_experience chỉ trích được ~45% bài đăng -> impute median
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')),
                      ('scaler', StandardScaler())]), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
])

---
## 1. Salary Model — chuẩn bị dữ liệu

- Chỉ dùng các dòng **có** salary (~30K)
- Loại outlier bằng IQR (giữ $15K–$500K)
- Chia **80/20** với `random_state=42` (tái lập được). Từ đây trở đi `X_test, y_test`
  được "niêm phong" cho đến bước đánh giá cuối cùng.

In [ ]:
salary_df = df.dropna(subset=['salary_annual']).copy()
Q1, Q3 = salary_df['salary_annual'].quantile([0.25, 0.75])
IQR = Q3 - Q1
lo, hi = max(Q1 - 1.5 * IQR, 15000), min(Q3 + 1.5 * IQR, 500000)
salary_df = salary_df[(salary_df['salary_annual'] >= lo) & (salary_df['salary_annual'] <= hi)]
print(f'Rows sau khi loại outlier: {len(salary_df):,}')

X = salary_df[features]
y = salary_df['salary_annual']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train):,} (80%) | Test cô lập: {len(X_test):,} (20%)')

---
## 2. So sánh thuật toán bằng Cross-Validation (chỉ trên TRAIN)

Chấm 4 ứng viên bằng **3-fold CV trên tập train** — tập test chưa hề được đụng tới:

1. `DummyRegressor(mean)` — baseline: nếu model không thắng nổi mức trung bình thì vô dụng
2. `LinearRegression` — baseline tuyến tính
3. `RandomForestRegressor` (mặc định)
4. `XGBRegressor`

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression

CV_FOLDS = 3
comparison = []

candidates_simple = {
    'Dummy (mean)': DummyRegressor(strategy='mean'),
    'LinearRegression': LinearRegression(),
    'RandomForest (default)': RandomForestRegressor(random_state=42, n_jobs=-1),
}
try:
    from xgboost import XGBRegressor
    candidates_simple['XGBoost'] = XGBRegressor(
        n_estimators=500, max_depth=8, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1, verbosity=0)
except ImportError:
    print('XGBoost chưa cài — bỏ qua.')

for name, model in candidates_simple.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=CV_FOLDS, scoring='r2', n_jobs=-1)
    comparison.append({'model': name, 'cv_r2_mean': scores.mean(), 'cv_r2_std': scores.std()})
    print(f'{name:<25} CV R² = {scores.mean():.4f} ± {scores.std():.4f}')

pd.DataFrame(comparison).sort_values('cv_r2_mean', ascending=False)

---
## 3. Tinh chỉnh siêu tham số + chọn model thắng cuộc (vẫn chỉ trên TRAIN)

- RandomForest: `GridSearchCV` 3-fold trên 12 tổ hợp tham số
- XGBoost: dùng điểm CV ở bước trên
- **Chọn model theo điểm CV trên train** — tuyệt đối không nhìn test set khi chọn

In [ ]:
candidates = {}

# XGBoost (nếu có) — lấy điểm CV từ bước so sánh
if 'XGBoost' in candidates_simple:
    pipe_xgb = Pipeline([('preprocessor', preprocessor), ('model', candidates_simple['XGBoost'])])
    cv_xgb = cross_val_score(pipe_xgb, X_train, y_train, cv=CV_FOLDS, scoring='r2', n_jobs=-1)
    pipe_xgb.fit(X_train, y_train)
    candidates['XGBoost'] = {'cv_mean': cv_xgb.mean(), 'cv_std': cv_xgb.std(), 'model': pipe_xgb}

# RandomForest tuned
pipe_rf = Pipeline([('preprocessor', preprocessor), ('model', RandomForestRegressor(random_state=42, n_jobs=-1))])
param_grid = {
    'model__n_estimators': [200, 400],
    'model__max_depth': [15, 25, None],
    'model__min_samples_leaf': [1, 3],
}
gs = GridSearchCV(pipe_rf, param_grid, cv=CV_FOLDS, scoring='r2', n_jobs=-1)
gs.fit(X_train, y_train)
candidates['RF'] = {'cv_mean': gs.best_score_,
                    'cv_std': float(gs.cv_results_['std_test_score'][gs.best_index_]),
                    'model': gs.best_estimator_}
print(f'RF tuned:  CV R² = {gs.best_score_:.4f}  (params: {gs.best_params_})')

best_model_name = max(candidates, key=lambda k: candidates[k]['cv_mean'])
best = candidates[best_model_name]
best_model = best['model']
print(f"\n>>> Model thắng theo CV: {best_model_name} (CV R² = {best['cv_mean']:.4f} ± {best['cv_std']:.4f})")

---
## 4. Đánh giá CUỐI CÙNG trên test cô lập — chỉ MỘT lần

Đây là lần **duy nhất** `X_test, y_test` được sử dụng. R²/MAE dưới đây là
độ chính xác báo cáo chính thức của mô hình trên dữ liệu chưa từng thấy.

In [ ]:
y_pred = best_model.predict(X_test)
r2_test = r2_score(y_test, y_pred)
mae_test = mean_absolute_error(y_test, y_pred)
print(f'FINAL — R² (test 20% cô lập) = {r2_test:.4f}')
print(f'FINAL — MAE (test 20% cô lập) = ${mae_test:,.0f}')

# Biểu đồ đánh giá trên TEST set (không phải in-sample)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(y_test, y_pred, alpha=0.3, s=10, c='skyblue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
axes[0].set_xlabel('Actual ($)'); axes[0].set_ylabel('Predicted ($)')
axes[0].set_title(f'Actual vs Predicted — TEST set (R²={r2_test:.3f})')

residuals = y_test - y_pred
axes[1].hist(residuals, bins=50, color='#0d6efd', edgecolor='white')
axes[1].set_xlabel('Residual ($)'); axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Residual Distribution — TEST set (MAE=${mae_test:,.0f})')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'salary_model_results.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 4b. Giải thích mô hình — Permutation Importance (trên test cô lập)

Xáo trộn từng feature trên **test set** và đo mức giảm R² — feature càng quan trọng,
R² giảm càng nhiều. Tính SAU khi đã chốt model nên không ảnh hưởng quy trình chọn model.
Trả lời trực tiếp câu hỏi phản biện: *"Yếu tố nào ảnh hưởng đến lương nhiều nhất?"*

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(best_model, X_test, y_test, n_repeats=5,
                              random_state=42, scoring='r2', n_jobs=-1)
importance = sorted(
    ({'feature': f, 'importance': round(float(m), 5)}
     for f, m in zip(features, perm.importances_mean)),
    key=lambda r: r['importance'], reverse=True)

top = importance[:12][::-1]
plt.figure(figsize=(9, 6))
plt.barh([r['feature'] for r in top], [r['importance'] for r in top], color='#00d2ff', alpha=0.7)
plt.xlabel('Permutation importance (drop in R², test set)')
plt.title('Salary Model — Feature Importance')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

pd.DataFrame(importance[:12])

In [ ]:
# Lưu model (compress=3 để file < 100MB — giới hạn GitHub) + metadata
joblib.dump(best_model, os.path.join(MODELS_DIR, 'best_salary_model.joblib'), compress=3)

meta_sal = {
    'feature_names': features,
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'mean_salary': float(y.mean()),
    'median_salary': float(y.median()),
    'r2_score': float(r2_test),          # điểm trên test CÔ LẬP
    'mae': float(mae_test),
    'cv_r2_mean': float(best['cv_mean']),  # điểm CV trên train (dùng để chọn model)
    'cv_r2_std': float(best['cv_std']),
    'cv_folds': CV_FOLDS,
    'train_size': len(X_train),
    'test_size': len(X_test),
    'model_type': best_model_name,
    'feature_importance': importance[:12],
    'salary_coverage': {
        'rows_total': int(len(df)),
        'rows_with_salary': int(df['salary_annual'].notna().sum()),
        'rows_range_based': int(df['salary_min'].notna().sum()),
        'rows_with_yoe': int(df['years_experience'].notna().sum()),
    },
    'it_domain': sorted(df['it_domain'].dropna().unique().tolist()),
    'seniority_level': sorted(df['seniority_level'].dropna().unique().tolist()),
    'job_type': sorted(df['job_type'].dropna().unique().tolist()),
    'state': sorted(df['state'].dropna().unique().tolist()),
}
joblib.dump(meta_sal, os.path.join(MODELS_DIR, 'salary_model_meta.joblib'))
print('Saved best_salary_model.joblib + salary_model_meta.joblib')

---
## 5. Demand Model — điểm nhu cầu tuyển dụng 0–100

**Thiết kế nhãn:** group theo `(it_domain, state, seniority_level, job_type)` → đếm số bài đăng
→ `log1p` (giảm skew) → scale 0–100.

**Cùng protocol:** 80/20, 5-fold CV trên train, đánh giá cuối trên test cô lập.

In [ ]:
demand_df = df.groupby(['it_domain', 'state', 'seniority_level', 'job_type']).size().reset_index(name='posting_count')
demand_df['demand_score'] = np.log1p(demand_df['posting_count'])
demand_df['demand_score'] = (demand_df['demand_score'] / demand_df['demand_score'].max() * 100).clip(0, 100)
print(f'Demand combos: {len(demand_df):,}')

X_dem = demand_df[['it_domain', 'state', 'seniority_level', 'job_type']]
y_dem = demand_df['demand_score']

preprocessor_dem = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
     ['it_domain', 'state', 'seniority_level', 'job_type'])
])

Xd_train, Xd_test, yd_train, yd_test = train_test_split(X_dem, y_dem, test_size=0.2, random_state=42)

pipe_dem = Pipeline([
    ('preprocessor', preprocessor_dem),
    ('model', RandomForestRegressor(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1)),
])
cv_dem = cross_val_score(pipe_dem, Xd_train, yd_train, cv=5, scoring='r2', n_jobs=-1)
print(f'Demand CV R² (5-fold, train) = {cv_dem.mean():.4f} ± {cv_dem.std():.4f}')

pipe_dem.fit(Xd_train, yd_train)
r2_dem = r2_score(yd_test, pipe_dem.predict(Xd_test))
print(f'Demand FINAL (test 20% cô lập) R² = {r2_dem:.4f}')

joblib.dump(pipe_dem, os.path.join(MODELS_DIR, 'demand_model.joblib'), compress=3)
meta_dem = {
    'model_type': 'RandomForestRegressor (Demand Score 0-100)',
    'r2_score': float(r2_dem),
    'cv_r2_mean': float(cv_dem.mean()),
    'cv_r2_std': float(cv_dem.std()),
    'cv_folds': 5,
    'max_posting_count': int(demand_df['posting_count'].max()),
    'it_domain': sorted(df['it_domain'].dropna().unique().tolist()),
    'seniority_level': sorted(df['seniority_level'].dropna().unique().tolist()),
    'job_type': sorted(df['job_type'].dropna().unique().tolist()),
    'state': sorted(df['state'].dropna().unique().tolist()),
}
joblib.dump(meta_dem, os.path.join(MODELS_DIR, 'demand_meta.joblib'))
print('Saved demand_model.joblib + demand_meta.joblib')

In [ ]:
# Biểu đồ demand cho dashboard
dom_df = df.groupby('it_domain').size().reset_index(name='count')
dom_df['score'] = np.log1p(dom_df['count']) / np.log1p(dom_df['count']).max() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(dom_df['it_domain'], dom_df['score'], color='#ffc107')
axes[0].set_title('Demand Score by Domain'); axes[0].set_xlabel('Demand Score (0-100)')
axes[1].barh(dom_df['it_domain'], dom_df['count'], color='#0d6efd')
axes[1].set_title('Posting Frequency by Domain'); axes[1].set_xlabel('Number of Postings')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '13_demand_radar.png'), dpi=150, bbox_inches='tight')
plt.savefig(os.path.join(FIGURES_DIR, '08_posting_frequency.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Cluster Model — KMeans + PCA (unsupervised)

Học **không giám sát** nên không cần train/test split — đánh giá bằng **silhouette score**.
- Elbow method để chọn K
- PCA giảm còn 5 chiều trước khi KMeans (dữ liệu one-hot ~370 chiều)

In [ ]:
cluster_df = df.dropna(subset=['salary_annual']).copy()
cluster_df = cluster_df[(cluster_df['salary_annual'] >= lo) & (cluster_df['salary_annual'] <= hi)]
X_cl = cluster_df[features]

# Elbow method trên không gian PCA
X_trans = preprocessor.fit_transform(X_cl)
pca5 = PCA(n_components=5, random_state=42)
X_pca = pca5.fit_transform(X_trans)

inertias = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_pca)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(K_range), inertias, 'o-', color='#00d2ff')
plt.axvline(5, color='#ffc107', linestyle='--', label='K=5 (chọn)')
plt.xlabel('K'); plt.ylabel('Inertia'); plt.title('Elbow Method')
plt.legend()
plt.savefig(os.path.join(FIGURES_DIR, 'elbow_method.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
pipe_cl = Pipeline([
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=5, random_state=42)),
    ('kmeans', KMeans(n_clusters=5, random_state=42, n_init=10)),
])
pipe_cl.fit(X_cl)

X_pca_final = pipe_cl.named_steps['pca'].transform(pipe_cl.named_steps['preprocessor'].transform(X_cl))
labels = pipe_cl.named_steps['kmeans'].labels_
sil = silhouette_score(X_pca_final, labels)
print(f'Silhouette score: {sil:.4f}')

cluster_desc = {}
for c in range(5):
    mask = labels == c
    subset = cluster_df[mask]
    top_domain = subset['it_domain'].mode().iloc[0] if len(subset) > 0 else 'N/A'
    top_sen = subset['seniority_level'].mode().iloc[0] if len(subset) > 0 else 'N/A'
    cluster_desc[c] = f"Cluster {c}: {top_domain} / {top_sen} / Avg ${subset['salary_annual'].mean():,.0f} ({mask.sum()} jobs)"
    print(' ', cluster_desc[c])

# Scatter 2 chiều PCA đầu tiên
plt.figure(figsize=(10, 7))
sc = plt.scatter(X_pca_final[:, 0], X_pca_final[:, 1], c=labels, cmap='tab10', s=6, alpha=0.5)
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title(f'KMeans K=5 (silhouette={sil:.3f})')
plt.colorbar(sc, label='Cluster')
plt.savefig(os.path.join(FIGURES_DIR, 'cluster_results.png'), dpi=150, bbox_inches='tight')
plt.show()

joblib.dump(pipe_cl, os.path.join(MODELS_DIR, 'cluster_model.joblib'), compress=3)
meta_cl = {
    'n_clusters': 5, 'pca_components': 5,
    'silhouette_score': float(sil),
    'cluster_descriptions': cluster_desc,
    'feature_names': features,
}
joblib.dump(meta_cl, os.path.join(MODELS_DIR, 'cluster_meta.joblib'))
print('Saved cluster_model.joblib + cluster_meta.joblib')

---
## 7. Tổng kết

| Model | Chọn/tinh chỉnh bằng | Độ chính xác cuối (test cô lập) |
|-------|----------------------|--------------------------------|
| Salary | 3-fold CV trên train (XGB vs RF-GridSearch) | R², MAE in ở Mục 4 |
| Demand | 5-fold CV trên train | R² in ở Mục 5 |
| Cluster | — (unsupervised) | Silhouette in ở Mục 6 |

Các con số này được lưu vào `models/*_meta.joblib` và hiển thị trên dashboard
(badge `R²` / `MAE` ở tab Dự Đoán Lương lấy từ `/api/meta`).

In [ ]:
print('=' * 60)
print('TỔNG KẾT ĐỘ CHÍNH XÁC (đánh giá trên test cô lập)')
print('=' * 60)
print(f"Salary : {best_model_name:<10} CV R²(train)={best['cv_mean']:.4f}  |  TEST R²={r2_test:.4f}, MAE=${mae_test:,.0f}")
print(f"Demand : RF          CV R²(train)={cv_dem.mean():.4f}  |  TEST R²={r2_dem:.4f}")
print(f"Cluster: KMeans K=5  silhouette={sil:.4f}")